# PUBG — Hai chế độ chạy trên Colab

Chọn `runtime` để chạy All-in-One không cần Drive, hoặc `drive` để lưu trực tiếp vào thư mục dự án trên Drive. Sau đó chạy từ trên xuống; bước 01 đọc ZIP theo batch và lưu Parquet nén, dùng lại shard hoàn tất khi chạy lại.

Ở chế độ runtime, kết quả là tạm thời; tải ZIP ở cell cuối trước khi ngắt phiên. Đây là cách chạy code hiện có, không phải chứng nhận đã hoàn tất mọi thí nghiệm trong đặc tả. Một số bước dùng pandas toàn bộ dữ liệu nên cần đủ RAM.


# 00 — Khởi tạo môi trường, cấu hình và trạng thái Checkpoint

**Mục tiêu:** Kiểm tra môi trường runtime, tài nguyên phần cứng (CPU/RAM/Disk), quyền ghi và tính tương thích của 10 file cấu hình.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
# @title Chọn nơi lưu dữ liệu { display-mode: "form" }
# @markdown `runtime`: không cần Drive, phù hợp notebook All-in-One.
# @markdown `drive`: lưu nối tiếp 13 notebook trong cùng thư mục Google Drive.
PUBG_STORAGE_MODE = "runtime"  # @param ["runtime", "drive"]
PUBG_DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/PUBG_Project/Project_PUBG"  # @param {type:"string"}
# @markdown Nhóm dùng cùng thư mục đã chia sẻ: bật True để tránh tạo nhầm project riêng khi thiếu shortcut.
PUBG_REQUIRE_EXISTING_PROJECT = False  # @param {type:"boolean"}
# @markdown Số dòng mỗi batch khi đọc CSV trong ZIP; giảm nếu RAM ít. Không lấy mẫu dữ liệu.
PUBG_BATCH_ROWS = 50000  # @param {type:"integer"}


In [ ]:
# Bootstrap: runtime mode needs no Drive; drive mode persists stage outputs.
import base64
import importlib.util
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PUBG_STORAGE_MODE = globals().get("PUBG_STORAGE_MODE", "runtime").strip().lower()
if PUBG_STORAGE_MODE not in {"runtime", "drive"}:
    raise ValueError("PUBG_STORAGE_MODE must be 'runtime' or 'drive'")

if PUBG_STORAGE_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Drive mode is available only on Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(globals().get(
        "PUBG_DRIVE_PROJECT_ROOT", "/content/drive/MyDrive/PUBG_Project/Project_PUBG"
    )).expanduser().resolve()
    if globals().get("PUBG_REQUIRE_EXISTING_PROJECT", False) and not (
        (PROJECT_ROOT / "configs/data.yaml").is_file()
        and (PROJECT_ROOT / "src/utils/config.py").is_file()
    ):
        raise FileNotFoundError(
            "Shared project not found: " + str(PROJECT_ROOT)
            + ". Check Editor access and the PUBG_Project shortcut in My Drive. "
            "No private project was created."
        )
else:
    _candidates = ([Path("/content/Project_PUBG")] if IN_COLAB else
                   [Path.cwd(), *Path.cwd().parents])
    _candidates += [p / "Project_PUBG" for p in list(_candidates)]
    PROJECT_ROOT = next((p.resolve() for p in _candidates
                         if (p / "configs/data.yaml").is_file() and (p / "src/utils/config.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "Project_PUBG"

if not (PROJECT_ROOT / "configs/data.yaml").is_file() or not (PROJECT_ROOT / "src/utils/config.py").is_file():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    _bundle = zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
    for _entry in _bundle.infolist():
        _target = (PROJECT_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Invalid bundled path")
        if not _target.exists():
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_bundle.read(_entry))
    _bundle.close()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if globals().get("PUBG_INSTALL_DEPENDENCIES", IN_COLAB) and not globals().get("_PUBG_PACKAGES_READY", False):
    _requirements = {
        "numpy": "numpy>=1.24.0", "pandas": "pandas>=2.0.0",
        "pyarrow": "pyarrow>=12.0.0", "duckdb": "duckdb>=0.9.0",
        "scipy": "scipy>=1.10.0", "sklearn": "scikit-learn>=1.3.0",
        "yaml": "pyyaml>=6.0",
    }
    _missing = [spec for module, spec in _requirements.items() if importlib.util.find_spec(module) is None]
    if _missing:
        print("Installing missing packages:", ", ".join(_missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", *_missing])
    _PUBG_PACKAGES_READY = True

if PUBG_STORAGE_MODE == "drive":
    os.environ["PUBG_SESSION_DRIVE_ROOT"] = str(PROJECT_ROOT)
    os.environ["PUBG_SESSION_TEMP_DIR"] = str(globals().get("PUBG_RUNTIME_TEMP_DIR", "/content/temp"))
else:
    os.environ.pop("PUBG_SESSION_DRIVE_ROOT", None)
    os.environ.pop("PUBG_SESSION_TEMP_DIR", None)
from src.utils.config import load_config, resolve_paths
cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
for _path in paths.values():
    _path.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
_PUBG_CELL_PROGRESS = {}  # Re-bootstrap invalidates prior cell state, even within one kernel.
print("Storage:", paths["data_root"], "| Results:", paths["reports_root"])
if PUBG_STORAGE_MODE == "drive":
    print("Storage mode: Google Drive. Stage outputs persist for the next notebook.")
else:
    print("Storage mode: runtime. No Drive authorization required; export before reset.")


### 1. Môi trường chạy

Mã nguồn đã được khởi tạo ở cell trên. Dữ liệu lưu theo chế độ runtime hoặc Drive đã chọn; kiểm tra đường dẫn được in bên dưới.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['00_setup.ipynb'] = -1
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
import os
from pathlib import Path
_pubg_progress['00_setup.ipynb'] = 5


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('00_setup.ipynb', -1) < 5:
    raise RuntimeError("00_setup.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['00_setup.ipynb'] = 5
# 1. Phát hiện môi trường thực thi
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("[Colab] Đang chạy trên Google Colab runtime.")
    print(f"Dữ liệu và kết quả: {paths['data_root']} | {paths['reports_root']}")
else:
    print("[Local] Đang chạy trên máy cục bộ.")
_pubg_progress['00_setup.ipynb'] = 6


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('00_setup.ipynb', -1) < 6:
    raise RuntimeError("00_setup.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['00_setup.ipynb'] = 6
# 2. Định vị thư mục gốc dự án và gắn vào sys.path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
_pubg_progress['00_setup.ipynb'] = 7


### 2. Nạp và kiểm tra tính toàn vẹn của 10 file cấu hình YAML

Đảm bảo tất cả các file cấu hình nghiệp vụ (`data.yaml`, `schema.yaml`, `paths.yaml`, v.v.) được nạp đầy đủ và thỏa mãn các ràng buộc schema trước khi thực thi.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('00_setup.ipynb', -1) < 7:
    raise RuntimeError("00_setup.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['00_setup.ipynb'] = 7
from src.utils.config import load_config, resolve_paths, validate_config

# Nạp toàn bộ 10 file config
cfg = load_config(str(PROJECT_ROOT / "configs"))
validate_config(cfg)

print("--- THÔNG TIN CẤU HÌNH NGHIÊN CỨU ---")
print(f"Môi trường kích hoạt: {cfg['paths']['active_environment']}")
print(f"Chế độ thực thi: {cfg['runtime']['mode']} (seed: {cfg['runtime']['random_state']})")
print(f"Dataset name: {cfg['data']['source']['dataset_name']}")
if cfg['data']['source'].get('archive_url'):
    print(f"Nguồn dữ liệu (Archive URL): {cfg['data']['source']['archive_url']}")
print(f"DuckDB Threads: {cfg['runtime']['duckdb']['threads']} | Memory Limit: {cfg['runtime']['duckdb']['memory_limit']}")
_pubg_progress['00_setup.ipynb'] = 9


### 3. Phân giải đường dẫn và khởi tạo cấu trúc thư mục lưu trữ (Paths Resolution & Storage Preparation)

Phân giải các đường dẫn logic (`raw_root`, `data_root`, `artifacts_root`, `reports_root`, v.v.) và tự động tạo tất cả các thư mục con phục vụ lưu trữ artifact và báo cáo.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('00_setup.ipynb', -1) < 9:
    raise RuntimeError("00_setup.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['00_setup.ipynb'] = 9
import pandas as pd

# Phân giải đường dẫn theo môi trường
paths = resolve_paths(cfg)

# Danh sách các thư mục cần khởi tạo sẵn
required_dirs = [
    ("raw_root", paths.get("raw_root", paths["raw"])),
    ("interim", paths["interim"]),
    ("processed", paths["processed"]),
    ("checkpoints", paths["checkpoints"]),
    ("experiments", paths["experiments"]),
    ("manifests", paths["manifests"]),
    ("models", paths["models"]),
    ("metrics", paths["metrics"]),
    ("logs", paths["logs"]),
    ("tables", paths["tables"]),
    ("figures", paths["figures"]),
    ("appendix", paths["appendix"]),
]

records = []
for name, d_path in required_dirs:
    existed = d_path.exists()
    d_path.mkdir(parents=True, exist_ok=True)
    records.append({
        "Thư mục logic": name,
        "Đường dẫn tuyệt đối": str(d_path),
        "Trạng thái": "Đã có sẵn" if existed else "Đã khởi tạo mới",
    })

df_paths = pd.DataFrame(records)
print("--- DANH MỤC ĐƯỜNG DẪN HỆ THỐNG ---")
for idx, row in df_paths.iterrows():
    print(f"[{row['Trạng thái']}] {row['Thư mục logic']:<15} -> {row['Đường dẫn tuyệt đối']}")

# Kiểm tra dữ liệu thô tại raw_root
raw_root = Path(paths["raw_root"]).resolve()
csv_shards = list(raw_root.glob("*.csv")) + list(raw_root.glob("*/*.csv"))
print(f"\nKiểm tra dữ liệu thô ({raw_root}):")
if csv_shards:
    print(f"  -> Đã tìm thấy {len(csv_shards)} file CSV thô sẵn sàng cho bước kiểm kê.")
else:
    print("  -> Chưa có CSV thô. Notebook 01 đọc ZIP theo batch, không giải nén toàn bộ ra đĩa.")
_pubg_progress['00_setup.ipynb'] = 11


### 4. Chẩn đoán tài nguyên phần cứng (CPU, RAM, Disk, GPU)

Kiểm tra dung lượng đĩa trống, quyền ghi và thông số phần cứng để đảm bảo an toàn bộ nhớ khi xử lý dữ liệu lớn.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('00_setup.ipynb', -1) < 11:
    raise RuntimeError("00_setup.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['00_setup.ipynb'] = 11
from src.utils.runtime import check_environment

env_report = check_environment(target_dir=paths["checkpoints"], min_disk_gb=5.0)

print("--- BÁO CÁO TÀI NGUYÊN PHẦN CỨNG ---")
print(f"Trạng thái hệ thống: {env_report['status'].upper()}")
print(f"Hệ điều hành: {env_report['runtime']['os_name']} {env_report['runtime']['os_release']}")
print(f"Phiên bản Python: {env_report['runtime']['python_version']}")
print(f"Số nhân CPU: {env_report['runtime']['cpu_count_logical']} (logical)")
print(f"Bộ nhớ RAM: {env_report['runtime'].get('available_ram_gb', 'N/A')} GB khả dụng / {env_report['runtime'].get('total_ram_gb', 'N/A')} GB tổng")
print(f"Dung lượng đĩa: {env_report['free_disk_gb']} GB trống / {env_report['total_disk_gb']} GB tổng")
print(f"Quyền ghi vào thư mục artifacts: {'HỢP LỆ' if env_report['can_write'] else 'KHÔNG CÓ QUYỀN'}")
if env_report['warnings']:
    print(f"Cảnh báo: {env_report['warnings']}")
_pubg_progress['00_setup.ipynb'] = 13


### 5. Kiểm tra trạng thái Checkpoint và DAG Run Readiness

Xác định xem hệ thống đã hoàn thành các stage nào từ các phiên chạy trước đó, sẵn sàng chuyển sang `01_download_validate.ipynb`.


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('00_setup.ipynb', -1) < 13:
    raise RuntimeError("00_setup.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['00_setup.ipynb'] = 13
from src.data.checkpoints import CheckpointManager

ckpt_mgr = CheckpointManager(manifest_path=paths["checkpoints"] / "checkpoint_manifest.json")
manifest = ckpt_mgr.load_manifest()

completed_stages = list(manifest.get("stages", {}).keys())
print("--- TRẠNG THÁI CHECKPOINT DAG ---")
if completed_stages:
    print(f"Các stage đã lưu checkpoint: {completed_stages}")
else:
    print("Chưa có checkpoint nào được lưu (Hệ thống ở trạng thái Clean Start).")

print("\n=> Sẵn sàng thực thi. Vui lòng mở và chạy notebook tiếp theo: '01_download_validate.ipynb'.")
_pubg_progress['00_setup.ipynb'] = 15


# 01 — Tải dữ liệu, kiểm kê Shards và xác thực Schema Contract

**Mục tiêu:** Đọc đầy đủ CSV trong ZIP theo batch, kiểm tra schema và lưu Parquet ZSTD. Không giải nén toàn bộ; khôi phục theo shard khi bị ngắt.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['01_download_validate.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('01_download_validate.ipynb', [])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection, atomic_write_json
from src.data.batch_ingest import ingest_sources, staged_paths
from src.data.checkpoints import CheckpointManager

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
if "con" in globals():
    con.close()
con = get_duckdb_connection(temp_dir=paths["temp_dir"], **{k: cfg["runtime"]["duckdb"][k] for k in ("memory_limit", "threads")})
ckpt_mgr = CheckpointManager(manifest_path=paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_progress['01_download_validate.ipynb'] = 4


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('01_download_validate.ipynb', -1) < 4:
    raise RuntimeError("01_download_validate.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['01_download_validate.ipynb'] = 4
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('01_download_validate.ipynb', [])
# 1. Đọc ZIP/CSV theo batch và ghi Parquet nén; không giải nén toàn bộ
staging_dir = paths["interim"] / "staging_shards"
inventory = ingest_sources(
    con, paths["raw_root"], staging_dir, cfg["data"], cfg["schema"],
    batch_rows=globals().get("PUBG_BATCH_ROWS", 50000),
    work_dir=paths["temp_dir"] / "batch_ingest",
)
atomic_write_json(paths["manifests"] / "source_inventory.json", inventory)
print(f"Đã xử lý đầy đủ {len(inventory['shards'])} shards, {sum(s['rows'] for s in inventory['shards']):,} dòng.")
_pubg_progress['01_download_validate.ipynb'] = 5


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('01_download_validate.ipynb', -1) < 5:
    raise RuntimeError("01_download_validate.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['01_download_validate.ipynb'] = 5
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('01_download_validate.ipynb', [])
# 2. Khóa checkpoint sau khi tất cả shard đã hoàn tất
for kind in ("aggregate", "deaths"):
    staged_paths(staging_dir, kind)  # Reject an incomplete manifest even if cell 1 failed.
ckpt_mgr.commit("schema", "schema_batch_v1", {"converted_agg_shards": staging_dir / "batch_manifest.json"})
print("Gate G1 Hoàn tất: Shards đã được kiểm kê và chuẩn hóa sang Parquet.")
_pubg_checkpoint.commit('notebook/01_download_validate.ipynb', "notebook_v1", {})

_pubg_progress['01_download_validate.ipynb'] = 6


# 02 — Chất lượng dữ liệu, cấu trúc Roster, Chronology và Khóa Split

**Mục tiêu:** Audit trùng lặp, kiểm tra roster, phân cấp Chronology Grade (A/B/C) và khóa Split Manifest trước khi phân tích quan hệ outcome (Gate G2).

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['02_data_quality_and_structure.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('02_data_quality_and_structure.ipynb', ['01_download_validate.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection, atomic_write_json
from src.data.cleaning import audit_and_clean_aggregate_data
from src.data.match_metadata import build_match_metadata
from src.models.splits import create_split_assignments
from src.analysis.eda import run_chronology_audit
from src.data.checkpoints import CheckpointManager

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
if "con" in globals():
    con.close()
con = get_duckdb_connection(temp_dir=paths["temp_dir"], **{k: cfg["runtime"]["duckdb"][k] for k in ("memory_limit", "threads")})
ckpt_mgr = CheckpointManager(manifest_path=paths["checkpoints"] / "checkpoint_manifest.json")

staging_dir = paths["interim"] / "staging_shards"
from src.data.batch_ingest import staged_paths
agg_shards = staged_paths(staging_dir, "aggregate")
if not agg_shards:
    raise FileNotFoundError(
        f"Không tìm thấy aggregate Parquet trong {staging_dir}. "
        "Hãy chạy xong notebook 01 với cùng PUBG_STORAGE_MODE và PUBG_DRIVE_PROJECT_ROOT."
    )
print(f"Đầu vào stage 02: {len(agg_shards)} aggregate shards từ {staging_dir}")
_pubg_progress['02_data_quality_and_structure.ipynb'] = 4


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('02_data_quality_and_structure.ipynb', -1) < 4:
    raise RuntimeError("02_data_quality_and_structure.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['02_data_quality_and_structure.ipynb'] = 4
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('02_data_quality_and_structure.ipynb', ['01_download_validate.ipynb'])
# 1. Làm sạch sơ bộ và ghi removal log
cleaned_pq = paths["interim"] / "cleaned_aggregate.parquet"
removal_csv = paths["tables"] / "removal_log.csv"
clean_summary = audit_and_clean_aggregate_data(con, agg_shards, cleaned_pq, removal_csv)
print(f"Làm sạch: Giữ lại {clean_summary['clean_rows']} dòng hợp lệ.")
_pubg_progress['02_data_quality_and_structure.ipynb'] = 5


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('02_data_quality_and_structure.ipynb', -1) < 5:
    raise RuntimeError("02_data_quality_and_structure.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['02_data_quality_and_structure.ipynb'] = 5
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('02_data_quality_and_structure.ipynb', ['01_download_validate.ipynb'])
# 2. Xây dựng Match Metadata (N_teams, duration proxy)
from src.data.match_metadata import build_match_metadata
cleaned_pq = paths["interim"] / "cleaned_aggregate.parquet"
if not cleaned_pq.is_file():
    raise FileNotFoundError(f"Chưa có dữ liệu sạch: {cleaned_pq}. Chạy cell làm sạch trước.")
meta_pq = paths["interim"] / "match_metadata.parquet"
total_matches = build_match_metadata(con, cleaned_pq, meta_pq)
_pubg_progress['02_data_quality_and_structure.ipynb'] = 6


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('02_data_quality_and_structure.ipynb', -1) < 6:
    raise RuntimeError("02_data_quality_and_structure.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['02_data_quality_and_structure.ipynb'] = 6
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('02_data_quality_and_structure.ipynb', ['01_download_validate.ipynb'])
# 3. Đánh giá Chronology Grade
meta_df = con.execute("SELECT match_date FROM read_parquet(?)", [str(meta_pq)]).df()
chrono_report = run_chronology_audit(meta_df)
print(f"Chronology Grade: {chrono_report['grade']} - {chrono_report.get('description', chrono_report.get('reason'))}")
atomic_write_json(paths["manifests"] / "chronology_report.json", chrono_report)
_pubg_progress['02_data_quality_and_structure.ipynb'] = 7


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('02_data_quality_and_structure.ipynb', -1) < 7:
    raise RuntimeError("02_data_quality_and_structure.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['02_data_quality_and_structure.ipynb'] = 7
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('02_data_quality_and_structure.ipynb', ['01_download_validate.ipynb'])
# 4. Khóa Split Assignments (Match isolation invariant)
split_pq = paths["interim"] / "split_assignments.parquet"
split_manifest = paths["manifests"] / "split_manifest.json"
split_meta = create_split_assignments(con, meta_pq, split_pq, split_manifest, strategy="group_by_match")

ckpt_mgr.commit("split_manifest", split_meta["config_hash"], {"split_assignments": split_pq, "match_metadata": meta_pq})
print("Gate G2 Hoàn tất: Split manifest đã được khóa trước EDA quan hệ.")
_pubg_checkpoint.commit('notebook/02_data_quality_and_structure.ipynb', "notebook_v1", {})

_pubg_progress['02_data_quality_and_structure.ipynb'] = 8


# 03 — Xây dựng Player-Match Base (Combat, Movement, Support, Placement)

**Mục tiêu:** Tính toán các đặc trưng hành vi người chơi và normalized placement với mẫu số đã xác minh.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['03_build_player_match.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('03_build_player_match.ipynb', ['02_data_quality_and_structure.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection
from src.data.checkpoints import CheckpointManager

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
if "con" in globals():
    con.close()
con = get_duckdb_connection(temp_dir=paths["temp_dir"], **{k: cfg["runtime"]["duckdb"][k] for k in ("memory_limit", "threads")})
ckpt_mgr = CheckpointManager(manifest_path=paths["checkpoints"] / "checkpoint_manifest.json")

print("Bước này được tích hợp liền mạch với Stage 04 trong quy trình streaming DuckDB.")
_pubg_checkpoint.commit('notebook/03_build_player_match.ipynb', "notebook_v1", {})

_pubg_progress['03_build_player_match.ipynb'] = 4


# 04 — Khai phá Combat Timing và hoàn thiện Player-Match Features

**Mục tiêu:** Tổng hợp các sự kiện hạ gục (Early, Mid, Late combat phases), loại trừ suicide/self-kill, và thực hiện Left Join bảo toàn số dòng vào player-match base.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['04_combat_timing.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('04_combat_timing.ipynb', ['01_download_validate.ipynb', '02_data_quality_and_structure.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection
from src.features.combat_timing import extract_and_aggregate_combat_timing, merge_player_match_and_timing
from src.data.checkpoints import CheckpointManager

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
if "con" in globals():
    con.close()
con = get_duckdb_connection(temp_dir=paths["temp_dir"], **{k: cfg["runtime"]["duckdb"][k] for k in ("memory_limit", "threads")})
ckpt_mgr = CheckpointManager(manifest_path=paths["checkpoints"] / "checkpoint_manifest.json")

staging_dir = paths["interim"] / "staging_shards"
from src.data.batch_ingest import staged_paths
death_shards = staged_paths(staging_dir, "deaths")
cleaned_agg = paths["interim"] / "cleaned_aggregate.parquet"
meta_pq = paths["interim"] / "match_metadata.parquet"
_pubg_progress['04_combat_timing.ipynb'] = 4


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('04_combat_timing.ipynb', -1) < 4:
    raise RuntimeError("04_combat_timing.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['04_combat_timing.ipynb'] = 4
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('04_combat_timing.ipynb', ['01_download_validate.ipynb', '02_data_quality_and_structure.ipynb'])
# 1. Trích xuất và tổng hợp thời điểm giao tranh
timing_pq = paths["interim"] / "combat_timing.parquet"
audit_summary = extract_and_aggregate_combat_timing(con, death_shards, meta_pq, timing_pq, paths["reports"] / "tables")
print(f"Tổng hợp Combat Timing: {audit_summary['valid_enemy_kills']} kills hợp lệ.")
_pubg_progress['04_combat_timing.ipynb'] = 5


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('04_combat_timing.ipynb', -1) < 5:
    raise RuntimeError("04_combat_timing.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['04_combat_timing.ipynb'] = 5
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('04_combat_timing.ipynb', ['01_download_validate.ipynb', '02_data_quality_and_structure.ipynb'])
# 2. Left join vào cơ sở dữ liệu player-match (Bảo toàn số dòng)
final_pq = paths["processed"] / "player_match_features.parquet"
discrepancy_csv = paths["reports"] / "tables" / "kill_discrepancy.csv"
final_rows = merge_player_match_and_timing(con, cleaned_agg, meta_pq, timing_pq, final_pq, discrepancy_csv)

ckpt_mgr.commit("player_match_features", "features_v1", {"final_dataset": final_pq})
print(f"Hoàn thành xuất player_match_features: {final_rows} dòng.")
_pubg_checkpoint.commit('notebook/04_combat_timing.ipynb', "notebook_v1", {})

_pubg_progress['04_combat_timing.ipynb'] = 6


# 05 — Phân bố đặc trưng và so sánh chế độ chơi

**Mục tiêu:** Tóm tắt phân bố đặc trưng và khác biệt Solo/Duo/Squad; lưu bảng thống kê và khuyến nghị chế độ phân tích.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['05_eda.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('05_eda.ipynb', ['04_combat_timing.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_parquet_df, atomic_write_json, atomic_write_csv
from src.analysis.eda import run_structural_eda, compute_distribution_summary
from src.analysis.mode_analysis import analyze_behavior_by_mode

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)

# Đọc dữ liệu đầy đủ cho EDA; không lấy mẫu
final_pq = paths["processed"] / "player_match_features.parquet"
df_sample = read_parquet_df(final_pq)

# Phase 3 & 4: Tóm tắt phân bố các đặc trưng
behavior_cols = [
    "player_kills", "player_dmg", "damage_per_kill",
    "player_dist_walk", "player_dist_ride", "total_distance", "walk_ratio",
    "player_assists", "player_dbno", "assist_ratio",
    "player_survive_time", "normalized_placement"
]
dist_summary = compute_distribution_summary(df_sample, behavior_cols)
atomic_write_csv(paths["tables"] / "data_quality_summary.csv", dist_summary)
print("--- TÓM TẮT PHÂN BỐ ĐẶC TRƯNG HÀNH VI ---")
print(dist_summary[["feature", "mean", "std", "median", "skewness", "zero_rate"]])

# Phase 5: Phân tích theo chế độ chơi
mode_res = analyze_behavior_by_mode(df_sample, behavior_cols)
atomic_write_csv(paths["tables"] / "mode_summary.csv", mode_res["summary_table"])
atomic_write_json(paths["manifests"] / "mode_analysis.json", {
    "mode_differences": mode_res["mode_differences"],
    "recommended_rq2_strategy": mode_res["recommended_rq2_strategy"],
})
print(f"Khuyến nghị chiến lược RQ2 Mode: {mode_res['recommended_rq2_strategy']}")
_pubg_checkpoint.commit('notebook/05_eda.ipynb', "notebook_v1", {})

_pubg_progress['05_eda.ipynb'] = 4


# 06 — Trả lời RQ1: Phân tích mối quan hệ giữa Hành vi, Thời điểm và Outcome

**Mục tiêu:** Tính toán tương quan Pearson và Spearman, tách biệt allowlist theo task, đánh giá sự khác biệt giữa các chế độ chơi và xuất rq1_relationship_summary.csv.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['06_rq1_analysis.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('06_rq1_analysis.ipynb', ['04_combat_timing.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_parquet_df
from src.features.registry import FeatureRegistry
from src.analysis.rq1 import run_rq1_analysis

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
registry = FeatureRegistry()

final_pq = paths["processed"] / "player_match_features.parquet"
df = read_parquet_df(final_pq)

rq1_table = run_rq1_analysis(df, registry, paths["tables"] / "rq1_relationship_summary.csv")
print("Top 10 mối quan hệ mạnh nhất với Normalized Placement:")
print(rq1_table[rq1_table["target"] == "normalized_placement"].sort_values(by="spearman_rho", ascending=False).head(10)[["feature", "group", "spearman_rho", "pearson_r", "is_primary_valid"]])
_pubg_checkpoint.commit('notebook/06_rq1_analysis.ipynb', "notebook_v1", {})

_pubg_progress['06_rq1_analysis.ipynb'] = 4


# 07 — Trả lời RQ2: Hồ sơ hành vi người chơi và Phân cụm (C1–C5)

**Mục tiêu:** Xây dựng Behavioral Profile (Design 3), chẩn đoán số cụm K tối ưu (Elbow/Silhouette/DB), thực thi C1-C5 và so sánh outcome sau phân cụm.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['07_rq2_clustering.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('07_rq2_clustering.ipynb', ['04_combat_timing.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_parquet_df
from src.features.profiles import build_player_behavioral_profiles, filter_profiles_by_retention
from src.data.io import atomic_write_csv
from src.analysis.clustering import run_k_diagnostics, execute_rq2_clustering, prepare_clustering_matrix

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)

final_pq = paths["processed"] / "player_match_features.parquet"
df = read_parquet_df(final_pq)
_pubg_progress['07_rq2_clustering.ipynb'] = 4


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('07_rq2_clustering.ipynb', -1) < 4:
    raise RuntimeError("07_rq2_clustering.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['07_rq2_clustering.ipynb'] = 4
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('07_rq2_clustering.ipynb', ['04_combat_timing.ipynb'])
# 1. Xây dựng hồ sơ hành vi người chơi
profiles, outcomes = build_player_behavioral_profiles(df)
filtered_profiles, filtered_outcomes = filter_profiles_by_retention(
    profiles, outcomes, min_games=cfg["rq2"].get("minimum_games_threshold") or 5)
_pubg_progress['07_rq2_clustering.ipynb'] = 5


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('07_rq2_clustering.ipynb', -1) < 5:
    raise RuntimeError("07_rq2_clustering.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['07_rq2_clustering.ipynb'] = 5
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('07_rq2_clustering.ipynb', ['04_combat_timing.ipynb'])
# 2. Chẩn đoán K
X = prepare_clustering_matrix(filtered_profiles, cfg["rq2"].get("scaler", "standard"))
k_diag = run_k_diagnostics(X, k_range=[2, 3, 4, 5, 6])
print("--- CHẨN ĐOÁN SỐ CỤM K ---")
print(k_diag)
atomic_write_csv(paths["tables"] / "k_diagnostics.csv", k_diag)
_pubg_progress['07_rq2_clustering.ipynb'] = 6


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('07_rq2_clustering.ipynb', -1) < 6:
    raise RuntimeError("07_rq2_clustering.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['07_rq2_clustering.ipynb'] = 6
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('07_rq2_clustering.ipynb', ['04_combat_timing.ipynb'])
# 3. Phân cụm chính thức C1 và đánh giá C2-C5
selected_k = cfg["rq2"]["n_clusters"] or 4
res = execute_rq2_clustering(filtered_profiles, filtered_outcomes, n_clusters=selected_k,
                             scaler_type=cfg["rq2"].get("scaler", "standard"), output_dir=paths["reports"] / "tables")
print("--- ĐỐI CHIẾU OUTCOME THEO CỤM (C5) ---")
print(res["outcome_comparison"])
_pubg_checkpoint.commit('notebook/07_rq2_clustering.ipynb', "notebook_v1", {})

_pubg_progress['07_rq2_clustering.ipynb'] = 7


# 08 — Xây dựng đặc trưng Lịch sử người chơi (Historical Features)

**Mục tiêu:** Xác minh thứ tự thời gian, tích lũy đặc trưng quá khứ (expanding window), đảm bảo không rò rỉ trận hiện tại hoặc tương lai.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['08_build_historical.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('08_build_historical.ipynb', ['02_data_quality_and_structure.ipynb', '04_combat_timing.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection, read_json, atomic_write_json
from src.features.historical import build_historical_features

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
if "con" in globals():
    con.close()
con = get_duckdb_connection(temp_dir=paths["temp_dir"], **{k: cfg["runtime"]["duckdb"][k] for k in ("memory_limit", "threads")})

chrono_report = read_json(paths["manifests"] / "chronology_report.json")
grade = chrono_report.get("grade", "Grade C")

hist_pq = paths["processed"] / "historical_player_match_features.parquet"
h_res = build_historical_features(con, paths["processed"] / "player_match_features.parquet", hist_pq, chronology_grade=grade)
atomic_write_json(paths["manifests"] / "historical_status.json", h_res)
print(f"Kết quả xây dựng lịch sử: {h_res}")
_pubg_checkpoint.commit('notebook/08_build_historical.ipynb', "notebook_v1", {})

_pubg_progress['08_build_historical.ipynb'] = 4


# 09 — RQ3: Dự đoán placement bằng Linear P1 và P2

**Mục tiêu:** Huấn luyện Linear có/không có survival trên Train, lưu dự đoán theo split và đánh giá Test. Chưa triển khai các thí nghiệm S1/S2/P3/T0/T1 trong notebook này.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['09_rq3_prediction.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('09_rq3_prediction.ipynb', ['02_data_quality_and_structure.ipynb', '04_combat_timing.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_parquet_df
from src.features.registry import FeatureRegistry
from src.models.baselines import TrainMeanRegressor, TrainMedianRegressor
from src.models.linear import LinearModelWrapper
from src.models.tree_models import HistGradientBoostingWrapper
from src.models.training import train_and_predict_experiment
from src.evaluation.metrics import compute_hierarchical_metrics

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
registry = FeatureRegistry()

final_pq = paths["processed"] / "player_match_features.parquet"
split_pq = paths["interim"] / "split_assignments.parquet"

df = read_parquet_df(final_pq)
splits = read_parquet_df(split_pq)
df = df.merge(splits[["match_id", "split"]], on="match_id", how="left", validate="many_to_one")

# Thí nghiệm P1 (với survival) vs P2 (bỏ survival trực tiếp)
p1_feats = registry.get_allowed_features("p1")
p2_feats = registry.get_allowed_features("p2")

print("Huấn luyện P2 (Linear Model - Không survival trực tiếp)...")
_, p2_preds = train_and_predict_experiment(df, p2_feats, "normalized_placement", LinearModelWrapper(model_type="exact"), "p2_linear", output_predictions_dir=paths["experiments"])

print("Huấn luyện P1 (Linear Model - Có survival trực tiếp)...")
_, p1_preds = train_and_predict_experiment(df, p1_feats, "normalized_placement", LinearModelWrapper(model_type="exact"), "p1_linear", output_predictions_dir=paths["experiments"])

p2_test = p2_preds[p2_preds["split"] == "test"]
p1_test = p1_preds[p1_preds["split"] == "test"]

print(f"P2 Test Micro MAE: {compute_hierarchical_metrics(p2_test)['micro']['mae']:.4f}")
print(f"P1 Test Micro MAE: {compute_hierarchical_metrics(p1_test)['micro']['mae']:.4f}")
_pubg_checkpoint.commit('notebook/09_rq3_prediction.ipynb', "notebook_v1", {})

_pubg_progress['09_rq3_prediction.ipynb'] = 4


# 10 — Đóng góp nhóm đặc trưng và phân tích sai số

**Mục tiêu:** Chạy group ablation cho Linear P2 và phân tích lát cắt sai số. Notebook hiện chưa tính khoảng tin cậy bootstrap hoặc thí nghiệm T0/T1.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['10_ablation_error_analysis.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('10_ablation_error_analysis.ipynb', ['09_rq3_prediction.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_parquet_df
from src.features.registry import FeatureRegistry
from src.evaluation.ablation import run_group_ablation_study
from src.evaluation.error_analysis import analyze_prediction_errors
from src.evaluation.bootstrap import run_paired_match_bootstrap

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
registry = FeatureRegistry()

final_pq = paths["processed"] / "player_match_features.parquet"
split_pq = paths["interim"] / "split_assignments.parquet"
df = read_parquet_df(final_pq).merge(read_parquet_df(split_pq)[["match_id", "split"]], on="match_id", how="left", validate="many_to_one")
_pubg_progress['10_ablation_error_analysis.ipynb'] = 4


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('10_ablation_error_analysis.ipynb', -1) < 4:
    raise RuntimeError("10_ablation_error_analysis.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['10_ablation_error_analysis.ipynb'] = 4
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('10_ablation_error_analysis.ipynb', ['09_rq3_prediction.ipynb'])
# 1. Group Ablation Study
p2_feats = registry.get_allowed_features("p2")
ablation_df = run_group_ablation_study(df, registry, p2_feats, "normalized_placement", paths["tables"] / "ablation_results.csv")
print("--- KẾT QUẢ GROUP ABLATION STUDY ---")
print(ablation_df[["ablation_experiment", "removed_group", "test_mae", "delta_mae_vs_full"]])
_pubg_progress['10_ablation_error_analysis.ipynb'] = 5


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('10_ablation_error_analysis.ipynb', -1) < 5:
    raise RuntimeError("10_ablation_error_analysis.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['10_ablation_error_analysis.ipynb'] = 5
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('10_ablation_error_analysis.ipynb', ['09_rq3_prediction.ipynb'])
# 2. Phân tích lát cắt sai số (Residual Error Analysis)
p2_preds = read_parquet_df(paths["experiments"] / "predictions_p2_linear.parquet")
err_df = analyze_prediction_errors(p2_preds, paths["tables"] / "error_analysis.csv")
print("--- PHÂN TÍCH SAI SỐ THEO TIERS ---")
print(err_df)
_pubg_checkpoint.commit('notebook/10_ablation_error_analysis.ipynb', "notebook_v1", {})

_pubg_progress['10_ablation_error_analysis.ipynb'] = 6


# 11 — Khóa kết quả nghiên cứu chính thức (Gate G5)

**Mục tiêu:** Xác nhận các run hợp lệ, tạo checksum SHA256 cho toàn bộ bảng biểu, mô hình và khóa final_results_manifest.json.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['11_finalize_results.ipynb'] = -1
from src.data.checkpoints import CheckpointManager
_pubg_checkpoint = CheckpointManager(paths["checkpoints"] / "checkpoint_manifest.json")
_pubg_checkpoint.begin_notebook('11_finalize_results.ipynb', ['06_rq1_analysis.ipynb', '07_rq2_clustering.ipynb', '09_rq3_prediction.ipynb', '10_ablation_error_analysis.ipynb'])
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.evaluation.finalize import build_final_results_manifest

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)

official_runs = {
    "rq1": "rq1_relationship_summary_v1",
    "p1": "p1_linear",
    "p2": "p2_linear",
    "ablation": "ablation_p2_groups",
}
import pandas as pd
cluster_table = paths["tables"] / "cluster_profile.csv"
if cluster_table.is_file():
    cluster_count = len(pd.read_csv(cluster_table))
    if cluster_count:
        official_runs["rq2"] = f"rq2_kmeans_k{cluster_count}"
for required in [paths["tables"] / "rq1_relationship_summary.csv",
                 paths["experiments"] / "predictions_p1_linear.parquet",
                 paths["experiments"] / "predictions_p2_linear.parquet",
                 paths["tables"] / "ablation_results.csv"]:
    if not required.is_file():
        raise FileNotFoundError(f"Chưa chạy xong các bước trước: {required}")

manifest = build_final_results_manifest(
    artifacts_root=paths["artifacts_root"],
    reports_root=paths["reports_root"],
    official_run_ids=official_runs,
    output_manifest_path=paths["manifests"] / "final_results_manifest.json"
)

print(f"Gate G5 Đã khóa: {len(manifest['tables'])} bảng kết quả chính thức đã được băm mã hóa bảo vệ.")
_pubg_checkpoint.commit('notebook/11_finalize_results.ipynb', "notebook_v1", {})

_pubg_progress['11_finalize_results.ipynb'] = 4


# 12 — Báo cáo tổng hợp Kết quả nghiên cứu (Chỉ đọc)

**Mục tiêu:** Tải trực tiếp từ final_results_manifest.json đã khóa. Tuyệt đối không huấn luyện lại, không gọi lại dữ liệu raw.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


In [ ]:
if "paths" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['12_final_results_summary.ipynb'] = -1
from src.data.io import read_json
_pubg_run_manifest = paths["checkpoints"] / "checkpoint_manifest.json"
if _pubg_run_manifest.is_file():
    _pubg_stages = read_json(_pubg_run_manifest).get("stages", {})
    if any(k.startswith("notebook/") and v.get("status") != "completed" for k, v in _pubg_stages.items()):
        raise RuntimeError("Có notebook chưa hoàn tất; chạy xong rồi khóa lại notebook 11 trước khi đọc kết quả.")
import gc
for _old_name in ('df', 'df_sample', 'df_paths', 'meta_df', 'splits', 'profiles', 'outcomes', 'filtered_profiles', 'filtered_outcomes', 'X', 'res', 'p1_preds', 'p2_preds', 'p1_test', 'p2_test', '_'):
    globals().pop(_old_name, None)
if 'con' in globals():
    globals().pop('con').close()
gc.collect()
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import read_json
from src.evaluation.finalize import verify_final_manifest_integrity

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)

manifest_path = paths["manifests"] / "final_results_manifest.json"
is_valid, mismatches = verify_final_manifest_integrity(manifest_path)

if not is_valid:
    raise ValueError(f"Checksum kết quả không khớp: {mismatches}")
else:
    print("XÁC THỰC THÀNH CÔNG: Toàn bộ bảng biểu và mô hình đều nguyên vẹn và khớp khóa bảo mật.")

manifest = read_json(manifest_path)
print(f"Thời điểm khóa kết quả: {manifest['finalized_at']}")
print(f"Các lần chạy chính thức: {manifest['official_run_ids']}")

# Hiển thị tóm tắt ablation study
abl_path = paths["tables"] / "ablation_results.csv"
if abl_path.is_file():
    print("--- ĐÓNG GÓP CỦA CÁC NHÓM BIẾN (ABLATION STUDY) ---")
    print(pd.read_csv(abl_path))
_pubg_progress['12_final_results_summary.ipynb'] = 4


## Tải kết quả về máy

ZIP mặc định chứa config, báo cáo và artifacts; không chứa raw/interim/processed. Bật `INCLUDE_DATA_CHECKPOINTS` nếu cần lưu dữ liệu để tiếp tục, ZIP có thể lớn.


In [ ]:
# Export results to your computer; no Drive authorization.
# Set True only when you also want the potentially large processed/interim data.
INCLUDE_DATA_CHECKPOINTS = False
import zipfile
from pathlib import Path

export_path = PROJECT_ROOT / "PUBG_results.zip"
roots = {"reports": paths["reports_root"], "figures": paths["figures"],
         "configs": PROJECT_ROOT / "configs", "artifacts": paths["artifacts_root"]}
if INCLUDE_DATA_CHECKPOINTS:
    roots.update({"data/interim": paths["interim"], "data/processed": paths["processed"]})
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for prefix, root in roots.items():
        for file in sorted(root.rglob("*")):
            relative = file.relative_to(root)
            if file.is_file() and not any(p in {"temp", "duckdb_temp", "__pycache__"} for p in relative.parts):
                archive.write(file, str(Path(prefix) / relative))
print("Export:", export_path, "bytes:", export_path.stat().st_size)
if IN_COLAB:
    from google.colab import files
    files.download(str(export_path))
else:
    from IPython.display import FileLink, display
    display(FileLink(str(export_path.relative_to(PROJECT_ROOT))))
